# TP 2 — Profiling & périmètre — NutriScope

> **Objectif :** connaître le jeu de données à fond et décider sur quoi porte NutriScope.

---

## 1. Profiling systématique des données

**Responsable : Sacha**

> Cette partie sera complétée à partir de l'analyse quantitative du dataset `food.parquet`.

### 1.1 Distributions

**À compléter :**

* Nombre total de produits :
* Distribution des principales variables numériques :
* Distribution du Nutri-Score :
* Distribution des catégories :
* Autres distributions pertinentes :

### 1.2 Cardinalités

**À compléter :**

* Cardinalité de `code` :
* Cardinalité des catégories :
* Cardinalité des marques :
* Cardinalité des autres variables importantes :

### 1.3 Doublons de codes-barres

**À compléter :**

* Nombre de codes-barres uniques :
* Nombre de codes-barres dupliqués :
* Taux de doublons :
* Conclusion :

### 1.4 Incohérences d'unités

**À compléter :**

* Énergies :
* Nutriments :
* Sel / sodium :
* Autres incohérences :

### 1.5 Valeurs impossibles / aberrantes

**À compléter :**

* `sugars_100g > 100` :
* Énergies nulles :
* Valeurs négatives :
* Valeurs extrêmement élevées :
* Autres anomalies :

### Conclusion du profiling

**À compléter par Sacha.**
---

# 2. Inventaire des colonnes

L'objectif est de déterminer quelles colonnes sont utiles aux différentes fonctionnalités de NutriScope et quelles colonnes constituent du bruit.

La classification a été réalisée selon les six usages :

* **Score** : prédiction/calcul du Nutri-Score ;
* **Substitution** : recherche d'un produit comparable et potentiellement meilleur ;
* **Images** : exploitation des images et classification visuelle ;
* **RAG** : informations utilisées par l'assistant ;
* **App** : recherche et affichage dans l'application ;
* **ML** : variables utilisables par les modèles de machine learning.

---

## 2.1 Colonnes d'identification et de catalogue

### À conserver

```text
code
product_name
generic_name
brands
brands_tags
categories
categories_tags
categories_fr
main_category
main_category_fr
quantity
url
```

### Justification

Ces informations permettent :

* d'identifier un produit ;
* de rechercher un produit ;
* de catégoriser le catalogue ;
* d'afficher les informations dans l'application ;
* d'alimenter le RAG ;
* de comparer des produits pour la substitution.

`code` est particulièrement important puisqu'il permet d'identifier le produit.

---

# 2.2 Données nutritionnelles

### À conserver

```text
nutriments
energy_100g
energy-kj_100g
energy-kcal_100g
proteins_100g
carbohydrates_100g
sugars_100g
fat_100g
saturated-fat_100g
fiber_100g
sodium_100g
salt_100g
fruits-vegetables-nuts_100g
```

Ainsi que les autres vitamines, minéraux et nutriments disponibles lorsqu'ils présentent une qualité de données suffisante.

### Justification

Ces données constituent le cœur du projet :

* prédiction du Nutri-Score ;
* comparaison nutritionnelle ;
* moteur de substitution ;
* RAG ;
* modèles ML.

---

# 2.3 Composition et ingrédients

### À conserver

```text
ingredients_text
traces
traces_tags

additives_n
additives
additives_tags

ingredients_from_palm_oil_n
ingredients_from_palm_oil
ingredients_from_palm_oil_tags

ingredients_that_may_be_from_palm_oil_n
ingredients_that_may_be_from_palm_oil
ingredients_that_may_be_from_palm_oil_tags
```

### Justification

Ces informations sont particulièrement utiles pour :

* le RAG ;
* le moteur de substitution ;
* la comparaison des produits ;
* l'analyse de la composition ;
* certains modèles NLP/ML.

---

# 2.4 Images

### À conserver

```text
image_url
image_small_url
```

### Justification

Ces colonnes sont nécessaires pour exploiter les photographies des produits et alimenter le futur **classifieur d'images entraîné avec Keras**.

Elles permettent également d'afficher les produits dans l'application.

---

# 2.5 Informations secondaires

### À conserver avec une priorité faible

```text
packaging
packaging_tags

labels
labels_tags
labels_fr

origins
origins_tags

manufacturing_places
manufacturing_places_tags

countries
countries_tags
countries_fr

serving_size
```

### Justification

Ces informations ne sont pas indispensables au calcul du Nutri-Score, mais peuvent être utiles pour :

* le RAG ;
* la substitution ;
* l'application ;
* la contextualisation des produits.

---

# 2.6 Colonnes considérées comme du bruit

### À supprimer

```text
creator
created_t
created_datetime
last_modified_t
last_modified_datetime

emb_codes
emb_codes_tags

first_packaging_code_geo

cities
cities_tags

purchase_places
stores
```

### Justification

Ces informations correspondent principalement à des métadonnées ou informations logistiques qui n'apportent pas de valeur directe aux fonctionnalités principales de NutriScope.

Elles peuvent donc être écartées afin de réduire le bruit et la complexité du dataset.

---

# 2.7 Cas particulier : Nutri-Score

Les colonnes :

```text
nutrition_grade_fr
nutriscore_grade
nutriscore_score
```

doivent être traitées séparément.

### Conservation

Elles sont conservées pour :

* mesurer la couverture du Nutri-Score ;
* analyser sa distribution ;
* constituer la cible ;
* évaluer les performances du modèle.

### Exclusion des features

Elles ne doivent **pas** être utilisées comme variables explicatives du modèle de prédiction du Nutri-Score.

### Justification

Ces colonnes contiennent directement le résultat que le modèle doit prédire, ou une information dérivée de celui-ci.

Les utiliser comme entrée créerait une **fuite de cible (target leakage)**.

---

# 3. Décision de périmètre en équipe

## 3.1 Rayons couverts au lancement

Après discussion, nous retenons **6 rayons de supermarché** :

1. **Boissons**
2. **Produits laitiers**
3. **Céréales et petit-déjeuner**
4. **Biscuits et snacks**
5. **Plats préparés et conserves**
6. **Sauces et condiments**

### Justification

Le choix de six rayons constitue un compromis entre **diversité et maîtrise du périmètre**.

Les rayons sélectionnés présentent des profils nutritionnels, des compositions et des caractéristiques visuelles relativement différents.

Cela permet notamment de tester :

* la prédiction du Nutri-Score sur des profils nutritionnels variés ;
* la reconnaissance d'images ;
* la classification des produits ;
* le moteur de substitution ;
* le RAG et les questions posées à l'assistant.

À l'inverse, couvrir un nombre trop important de rayons augmenterait :

* la complexité de la classification ;
* le risque de chevauchement entre catégories ;
* les ambiguïtés dans la détection d'un produit ;
* la difficulté d'obtenir suffisamment de données homogènes pour chaque classe.

Le choix définitif des catégories Open Food Facts correspondant à chaque rayon devra être validé à partir du profiling du dataset.

---

## 3.2 Colonnes conservées

La sélection des colonnes repose sur quatre critères :

1. **Utilité métier**
2. **Qualité des données**
3. **Complétude**
4. **Risque de fuite de cible**

Les colonnes sont donc réparties en :

```text
KEEP
KEEP_LOW
ANALYSIS_ONLY
DROP
```

### KEEP

Colonnes indispensables ou fortement utiles à une fonctionnalité.

### KEEP_LOW

Colonnes utiles mais secondaires.

### ANALYSIS_ONLY

Colonnes conservées pour l'analyse ou l'évaluation mais non utilisées comme features.

### DROP

Colonnes considérées comme du bruit ou sans utilité pour le périmètre retenu.

---

# 3.3 Seuil de complétude minimal par produit

**À définir après le profiling de Sacha.**

Le seuil devra être défini **en fonction de la fonctionnalité**, car toutes les fonctionnalités n'ont pas les mêmes besoins.

Par exemple :

### Modèle Nutri-Score

Un produit doit disposer des principales informations nutritionnelles nécessaires au modèle.

```text
Énergie
Sucres
Acides gras saturés
Sel / sodium
...
```

### RAG

Un produit doit disposer d'un minimum d'informations descriptives permettant à l'assistant de répondre de manière pertinente.

### Images

Un produit doit disposer d'une image exploitable associée à son identifiant.

### Décision finale

**À compléter après le profiling :**

> Seuil de complétude retenu : **XX %**

> Justification : **à compléter selon les résultats du profiling.**

---

# 4. Perimetre
> voir le document docs/perimetre.md


---

# Synthèse de notre décision

À ce stade, votre décision peut être résumée ainsi :

```text
                    NutriScope
                        │
       ┌────────────────┼────────────────┐
       │                │                │
   6 RAYONS         DONNÉES           QUALITÉ
       │             UTILES              │
       │                │                │
       ↓                ↓                ↓
 Boissons          Nutrition       Profiling
 Laitiers          Ingrédients     Complétude
 Céréales          Catégories      Cardinalité
 Biscuits          Marques         Doublons
 Plats préparés    Images          Anomalies
 Sauces            etc.
       │                │                │
       └────────────────┼────────────────┘
                        ↓
                 PÉRIMÈTRE FINAL
                        ↓
                docs/perimetre.md
```

### Ce qui reste volontairement à compléter

**Sacha :**

* tout le **§1 Profiling** ;
* les chiffres réels de complétude ;
* les distributions ;
* cardinalités ;
* doublons ;
* incohérences d'unités ;
* valeurs impossibles ;
* éléments nécessaires pour fixer le seuil de complétude.

**Toi / équipe :**

* validation définitive des 6 rayons ;
* association des catégories Open Food Facts à ces 6 rayons ;
* validation de la matrice des colonnes ;
* décision finale sur le seuil de complétude une fois les résultats de Sacha disponibles ;
* rédaction finale de `docs/perimetre.md`.

**Donc votre travail est maintenant correctement séparé : Sacha produit les preuves quantitatives, et toi tu peux avancer sur le périmètre fonctionnel et les décisions de sélection.**
